
# Autocallable – Advanced Playground (Memory, Worst‑of, Toy Greeks)

This notebook extends the basic playground with:
- **Coupon memory** (missed coupons paid upon early call)
- **Worst‑of basket** (path check & terminal based on worst underlying)
- **Toy sensitivities (Greeks)** via finite differences (didactic; not a pricing model)

> ⚠️ Educational only. This ignores real-world pricing (stochastic paths, discounting, smiles, barriers dynamics, discrete dividends, calendars, etc.).


In [ ]:

# === Global Parameters ===
N = 100.0      # nominal
S0 = 100.0     # initial level for scaling ratios (per underlying)
Ba = 1.00      # autocall barrier (fraction of initial), e.g. 1.00 = 100%
Bp = 0.60      # protection barrier at maturity, e.g. 0.60 = 60%
c  = 0.10      # coupon per observation period
years = 5      # max tenor (observation frequency = annual for simplicity)

coupon_memory = True   # if True, coupons accumulate: c * year_index at call
worst_of_basket = True # if True, use worst-of across underlyings for checks & terminal

# Basket configuration
n_underlyings = 3
# Terminal ratios S_T / S0 for each underlying (for single-scenario evaluation)
R_terminals = [0.85, 1.02, 0.75]

# Grid for plotting (assumes symmetric basket or uses the first underlying if gradient plotted 1D)
r_min, r_max, r_step = 0.0, 1.8, 0.0025

div_yield = 0.02  # used only for equity comparison visuals


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline



## Core Helpers
- `worst_ratio_at_obs(R_vec, i)` – geometric interpolation to observation `i` and worst-of across names
- `payoff_autocall(R_vec)` – payoff using worst-of + coupon memory flags


In [ ]:

def worst_ratio_at_obs(R_vec, i, T=5):
    """Geometric interpolation: ratio at obs i for each name is R_j ** (i/T).
    Returns worst-of across names.
    """
    if isinstance(R_vec, (int, float)):
        R_vec = [float(R_vec)]
    ratios = [max(0.0, float(r)) ** (i / T) for r in R_vec]
    return min(ratios)

def payoff_autocall(R_vec):
    """Autocall payoff with options:
       - worst_of_basket: using worst-of across names for both barrier checks & terminal
       - coupon_memory: coupon = c * year_index at call; else just c once if not
    """
    if not isinstance(R_vec, (list, tuple, np.ndarray)):
        R_vec = [float(R_vec)]
    R_vec = [max(0.0, float(r)) for r in R_vec]

    # Observation times 1..years
    obs_times = range(1, years + 1)
    ac_idx = None
    for i in obs_times:
        wo_i = worst_ratio_at_obs(R_vec, i, T=years) if worst_of_basket else min([r ** (i/years) for r in R_vec])
        if wo_i >= Ba:
            ac_idx = i
            break

    if ac_idx is not None:
        if coupon_memory:
            return N * (1.0 + c * ac_idx)
        else:
            return N * (1.0 + c)

    # No autocall: evaluate at maturity on worst-of
    R_T = min(R_vec) if worst_of_basket else min(R_vec)
    if R_T >= Bp:
        return N
    else:
        return N * R_T



## 1D Plot (assuming symmetric basket or single name)
We sweep a single terminal ratio `R` and apply it to all names (worst-of collapses to `R`).


In [ ]:

R_grid = np.arange(r_min, r_max + r_step, r_step)
payoffs = np.array([payoff_autocall([r]*n_underlyings) for r in R_grid])
equity_with_div = N * R_grid * (1 + div_yield * years)
bond_line = np.full_like(R_grid, N)

plt.figure(figsize=(8,5))
plt.plot(R_grid*100, payoffs, label=f"Autocallable (memory={coupon_memory}, worst-of={worst_of_basket})")
plt.plot(R_grid*100, equity_with_div, "--", label=f"Equity (+{div_yield:.0%}/y dividends)")
plt.plot(R_grid*100, bond_line, ":", label="Bond (capital)")
plt.axvline(Bp*100, linestyle=":", color="red", label=f"Protection {Bp:.0%}")
plt.axvline(Ba*100, linestyle=":", color="gray", label=f"Autocall {Ba:.0%}")
plt.xlabel("Terminal performance (applied to each name) S_T/S0 (%)")
plt.ylabel("Redemption / Payoff")
plt.title("Autocallable – 1D sweep (symmetric basket)")
plt.legend()
plt.tight_layout()
plt.show()



## Scenario Table (asymmetric basket)
Provide different terminal ratios per name in `R_terminals` and see payoff + earliest autocall year under the geometric path proxy.


In [ ]:

def earliest_autocall_year_vector(R_vec):
    for i in range(1, years+1):
        wo = worst_ratio_at_obs(R_vec, i, T=years) if worst_of_basket else min([r ** (i/years) for r in R_vec])
        if wo >= Ba:
            return i
    return None

scenarios = [
    [0.50, 1.10, 0.95],
    [0.65, 0.92, 0.88],
    [0.95, 0.98, 1.01],
    [1.02, 1.05, 0.99],
    [0.55, 0.58, 0.62],
    R_terminals
]
rows = []
for vec in scenarios:
    rows.append({
        "R vector": [f"{r:,.0%}" for r in vec],
        "Earliest AC year": earliest_autocall_year_vector(vec) or "No AC",
        "Payoff": round(payoff_autocall(vec), 2)
    })
pd.DataFrame(rows)



## Toy Greeks (Finite Differences)
We compute **partial derivatives** of the payoff w.r.t. each terminal ratio `R_j` (gradient).
This is **not** a trading delta: it's a static slope of the terminal payoff vs. terminal level (didactic).


In [ ]:

def finite_diff_gradient(R_vec, h=1e-4):
    R_vec = [float(r) for r in R_vec]
    base = payoff_autocall(R_vec)
    grads = []
    for j in range(len(R_vec)):
        bumped = R_vec.copy()
        bumped[j] = max(0.0, bumped[j] + h)
        f_plus = payoff_autocall(bumped)
        grads.append((f_plus - base) / h)
    return np.array(grads), base

grads, base_payoff = finite_diff_gradient(R_terminals, h=1e-5)
pd.DataFrame({
    "Underlying": [f"Name {i+1}" for i in range(len(R_terminals))],
    "R_T": [f"{r:,.2%}" for r in R_terminals],
    "Toy dPayoff/dR_T": grads
})



## Vega-like Sensitivity (Heuristic)
There's no volatility in this toy model. As a placeholder, we approximate a **'vega-like'** sensitivity by applying a small **dispersion bump** to the path proxy:
- We increase curvature of the geometric path (simulate ‘higher vol’ causing lower early levels) and recompute the earliest call check.
This is purely didactic.


In [ ]:

def payoff_with_curvature(R_vec, kappa=1.0):
    """kappa > 1 bends the path down: r_i = R^( (i/T)^kappa ).
       kappa < 1 bends it up. kappa = 1 is the base case.
    """
    if not isinstance(R_vec, (list, tuple, np.ndarray)):
        R_vec = [float(R_vec)]
    R_vec = [max(0.0, float(r)) for r in R_vec]
    # observation with curvature
    ac_idx = None
    for i in range(1, years+1):
        powi = (i/years)**kappa
        ratios = [r**powi for r in R_vec]
        wo_i = min(ratios) if worst_of_basket else min(ratios)
        if wo_i >= Ba:
            ac_idx = i; break
    if ac_idx is not None:
        return N * (1.0 + c * (ac_idx if coupon_memory else 1.0))
    R_T = min(R_vec) if worst_of_basket else min(R_vec)
    return N if R_T >= Bp else N * R_T

def vega_like(R_vec, dk=1e-3):
    base = payoff_with_curvature(R_vec, kappa=1.0)
    up = payoff_with_curvature(R_vec, kappa=1.0 + dk)
    return (up - base) / dk, base

vl, base_payoff_curved = vega_like(R_terminals, dk=1e-3)
print("Vega-like (dPayoff/dkappa) at scenario R_terminals:", vl)
print("Base payoff (kappa=1):", base_payoff_curved)
